# 🩺 Skin Lesion Classification — Full Architecture
### ANFIS + N-Gram + BiLSTM + Transformer + GWO-CNN
**Optimized for Local GPU Server** | HAM10000 Dataset (Kaggle)

**Targets:** Accuracy, Precision, Recall, F1, Sensitivity, Specificity ≥ 94–95%

## ⚙️ Step 1 — Install Dependencies (run once)

In [ ]:
import subprocess, sys

pkgs = [
    "kaggle", "torch", "torchvision", "scikit-learn",
    "matplotlib", "seaborn", "pandas", "numpy", "tqdm", "Pillow", "opencv-python"
]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", p, "-q"])
print("✅ All packages installed!")


## 🔑 Step 2 — Kaggle API Credentials & Dataset Download

In [ ]:
import os, json

# ── Paste your Kaggle credentials here ──────────────────────
KAGGLE_USERNAME = "your_kaggle_username"     # ← REPLACE
KAGGLE_KEY      = "your_kaggle_api_key"      # ← REPLACE
# ────────────────────────────────────────────────────────────

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
cred_path = os.path.expanduser("~/.kaggle/kaggle.json")
with open(cred_path, "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(cred_path, 0o600)
print(f"✅ Kaggle credentials saved to {cred_path}")

DATA_DIR     = "./ham10000"
DATASET_SLUG = "kmader/skin-lesion-analysis-toward-melanoma-detection"
os.makedirs(DATA_DIR, exist_ok=True)

csv_path = os.path.join(DATA_DIR, "HAM10000_metadata.csv")
if not os.path.exists(csv_path):
    print("⬇️  Downloading HAM10000 (~2.4 GB) from Kaggle ...")
    ret = os.system(f"kaggle datasets download -d {DATASET_SLUG} -p {DATA_DIR} --unzip")
    if ret == 0:
        print("✅ Download & extraction complete!")
    else:
        print("❌ Download failed — check your credentials or network")
else:
    print("✅ Dataset already present, skipping download")

print("\n📁 Dataset contents:")
for item in sorted(os.listdir(DATA_DIR)):
    full = os.path.join(DATA_DIR, item)
    size = os.path.getsize(full) if os.path.isfile(full) else "dir"
    print(f"   {item}  ({size})")


## 📚 Step 3 — Imports & GPU Configuration

In [ ]:
import os, math, random, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, auc,
    classification_report
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# ── GPU Setup ──────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device : {DEVICE}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"   GPUs available : {n_gpus}")
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / 1024**3
        print(f"   GPU {i}: {props.name}  |  VRAM: {mem_gb:.1f} GB")
    torch.backends.cudnn.benchmark = True   # speed up fixed-size inputs
    torch.backends.cudnn.deterministic = False

# ── Hyperparameters (GPU-server optimized) ──────────────────
DATA_DIR    = "./ham10000"
IMG_SIZE    = 224          # higher res for GPU server
BATCH_SIZE  = 64           # larger batch for GPU
EPOCHS      = 40
NUM_CLASSES = 7
CLASS_NAMES = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
LESION_MAP  = {c: i for i, c in enumerate(CLASS_NAMES)}
NUM_WORKERS = min(8, os.cpu_count())   # max parallel workers
PIN_MEMORY  = True

print(f"\n⚙️  Config:")
print(f"   IMG_SIZE   = {IMG_SIZE}")
print(f"   BATCH_SIZE = {BATCH_SIZE}")
print(f"   EPOCHS     = {EPOCHS}")
print(f"   NUM_WORKERS= {NUM_WORKERS}")
print("✅ Ready!")


## 🗂️ Step 4 — Load & Explore Dataset

In [ ]:
def build_dataframe(data_dir):
    meta = pd.read_csv(os.path.join(data_dir, "HAM10000_metadata.csv"))
    # Recursively find all images
    img_map = {}
    for root, dirs, files in os.walk(data_dir):
        for fname in files:
            if fname.lower().endswith(('.jpg','.jpeg','.png')):
                key = os.path.splitext(fname)[0]
                img_map[key] = os.path.join(root, fname)
    meta["path"] = meta["image_id"].map(img_map)
    meta = meta.dropna(subset=["path"])
    return meta

df = build_dataframe(DATA_DIR)

print(f"📊 Total images : {len(df)}")
print(f"   Columns      : {list(df.columns)}\n")
print("Class distribution:")
vc = df["dx"].value_counts()
for cls, cnt in vc.items():
    bar = '█' * (cnt // 100)
    print(f"  {cls:6s}: {cnt:5d}  {bar}")

# Visualise class balance
fig, ax = plt.subplots(figsize=(10, 4))
vc.plot(kind='bar', ax=ax, color=plt.cm.Set2(np.linspace(0,1,7)), edgecolor='white')
ax.set_title("HAM10000 Class Distribution", fontsize=14, fontweight='bold')
ax.set_xlabel("Lesion Type"); ax.set_ylabel("Count")
ax.tick_params(axis='x', rotation=0)
for p in ax.patches:
    ax.text(p.get_x()+p.get_width()/2, p.get_height()+30,
            str(int(p.get_height())), ha='center', fontsize=10)
plt.tight_layout(); plt.show()


## 🔄 Step 5 — Dataset Class & DataLoaders with Weighted Sampling

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = LESION_MAP[row["dx"]]
        img   = Image.open(row["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


def make_weighted_sampler(df):
    labels  = [LESION_MAP[d] for d in df["dx"]]
    counts  = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    weights = 1.0 / counts
    sample_w = [weights[l] for l in labels]
    return WeightedRandomSampler(sample_w, len(sample_w), replacement=True)


def get_loaders(df):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    # Aggressive augmentation for GPU server
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(30),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.RandomGrayscale(p=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE+16, IMG_SIZE+16)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_df, test_df = train_test_split(df, test_size=0.20, stratify=df["dx"], random_state=42)
    train_df, val_df  = train_test_split(train_df, test_size=0.10, stratify=train_df["dx"], random_state=42)

    sampler = make_weighted_sampler(train_df)

    train_loader = DataLoader(
        SkinLesionDataset(train_df, train_tf),
        batch_size=BATCH_SIZE, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        prefetch_factor=2, persistent_workers=True
    )
    val_loader = DataLoader(
        SkinLesionDataset(val_df, val_tf),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        prefetch_factor=2, persistent_workers=True
    )
    test_loader = DataLoader(
        SkinLesionDataset(test_df, val_tf),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        prefetch_factor=2, persistent_workers=True
    )

    print(f"✅ Train : {len(train_df):,}  |  Val : {len(val_df):,}  |  Test : {len(test_df):,}")
    return train_loader, val_loader, test_loader, test_df

train_loader, val_loader, test_loader, test_df = get_loaders(df)

# Show sample batch
imgs, labels = next(iter(train_loader))
print(f"Batch shape : {imgs.shape}  Labels : {labels[:8].tolist()}")


## 🧠 Step 6 — Full Architecture: ANFIS + N-Gram + BiLSTM + Transformer + GWO-CNN

In [ ]:
# ─────────────────────────────────────────
# 6A. ANFIS (Adaptive Neuro-Fuzzy Inference)
# ─────────────────────────────────────────
class ANFISLayer(nn.Module):
    """Takagi-Sugeno ANFIS with Gaussian membership functions."""
    def __init__(self, in_features, n_rules=16):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(n_rules, in_features) * 0.1)
        self.sigmas  = nn.Parameter(torch.ones(n_rules,  in_features))
        self.linear  = nn.Linear(in_features, n_rules)
        self.out_proj = nn.Linear(n_rules, in_features)

    def forward(self, x):
        mu    = torch.exp(-0.5 * ((x.unsqueeze(1) - self.centers) / (self.sigmas.abs() + 1e-6))**2)
        w     = mu.prod(dim=-1)                          # (B, R)
        w_bar = w / (w.sum(dim=1, keepdim=True) + 1e-6) # normalised firing
        f     = self.linear(x)                           # (B, R)
        out   = (w_bar * f)                              # (B, R)
        return self.out_proj(out)                        # (B, F)


# ─────────────────────────────────────────
# 6B. N-Gram Visual Patch Extractor
# ─────────────────────────────────────────
class NGramPatchExtractor(nn.Module):
    """Visual n-grams: overlapping 1-D convolutions on flattened spatial features."""
    def __init__(self, in_channels=2048, patch_dim=512, n=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, patch_dim, kernel_size=1)
        self.conv3 = nn.Conv1d(in_channels, patch_dim, kernel_size=n, padding=n//2)
        self.conv5 = nn.Conv1d(in_channels, patch_dim, kernel_size=5, padding=2)
        self.proj  = nn.Linear(patch_dim * 3, patch_dim)
        self.norm  = nn.LayerNorm(patch_dim)
        self.drop  = nn.Dropout(0.1)

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.view(B, C, H * W)            # (B, C, S)
        g1 = self.conv1(x).permute(0,2,1)  # (B, S, D)
        g3 = self.conv3(x).permute(0,2,1)
        g5 = self.conv5(x).permute(0,2,1)
        g  = torch.cat([g1, g3, g5], dim=-1)
        return self.drop(self.norm(self.proj(g)))  # (B, S, patch_dim)


# ─────────────────────────────────────────
# 6C. BiLSTM Dual-Stream Fusion
# ─────────────────────────────────────────
class BiLSTMStream(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=0.3)
        self.out_dim = hidden_dim * 2

    def forward(self, x):
        out, _ = self.lstm(x)
        # Global average + max pooling over time
        avg = out.mean(dim=1)
        mx  = out.max(dim=1).values
        return avg + mx   # (B, hidden*2)


class DualStreamFusion(nn.Module):
    """Two independent BiLSTM streams fused with gating."""
    def __init__(self, input_dim=512, hidden_dim=256):
        super().__init__()
        self.stream1 = BiLSTMStream(input_dim, hidden_dim)
        self.stream2 = BiLSTMStream(input_dim, hidden_dim)
        fused = hidden_dim * 4
        self.gate   = nn.Sequential(nn.Linear(fused, fused), nn.Sigmoid())
        self.fusion = nn.Sequential(
            nn.Linear(fused, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(0.3)
        )

    def forward(self, x):
        s1 = self.stream1(x)
        s2 = self.stream2(x)
        cat = torch.cat([s1, s2], dim=-1)
        return self.fusion(cat * self.gate(cat))  # (B, hidden*2)


# ─────────────────────────────────────────
# 6D. Transformer Encoder with Bio-Inspired Attention
# ─────────────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, d=512, heads=8, ff=2048, drop=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True)
        self.ff    = nn.Sequential(
            nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d)
        )
        self.n1    = nn.LayerNorm(d)
        self.n2    = nn.LayerNorm(d)
        self.drop  = nn.Dropout(drop)

    def forward(self, x):
        a, _ = self.attn(x, x, x)
        x    = self.n1(x + self.drop(a))
        return self.n2(x + self.drop(self.ff(x)))


class TransformerAttentionModel(nn.Module):
    def __init__(self, d=512, heads=8, layers=4):
        super().__init__()
        self.pos_enc = nn.Parameter(torch.randn(1, 196+1, d) * 0.02)  # learnable PE
        self.cls_tok = nn.Parameter(torch.zeros(1, 1, d))
        self.blocks  = nn.ModuleList([TransformerBlock(d, heads) for _ in range(layers)])
        self.norm    = nn.LayerNorm(d)

    def forward(self, x):
        B = x.shape[0]
        cls = self.cls_tok.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = x + self.pos_enc[:, :x.shape[1], :]
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x[:, 0])   # CLS token → (B, d)


# ─────────────────────────────────────────
# 6E. GWO-Optimised CNN Backbone (ResNet50 + SE attention)
# ─────────────────────────────────────────
class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, ch, r=16):
        super().__init__()
        self.sq = nn.AdaptiveAvgPool2d(1)
        self.ex = nn.Sequential(
            nn.Flatten(),
            nn.Linear(ch, ch//r), nn.ReLU(),
            nn.Linear(ch//r, ch), nn.Sigmoid()
        )

    def forward(self, x):
        w = self.ex(self.sq(x)).view(x.shape[0], -1, 1, 1)
        return x * w


class GWOCNNBackbone(nn.Module):
    """ResNet50 with SE blocks; GWO decides which layers to unfreeze."""
    def __init__(self, unfreeze_layers=2):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.stem   = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.se     = SEBlock(2048, r=16)
        self.out_ch = 2048

        # Freeze all, then selectively unfreeze last N layers (GWO heuristic)
        for p in self.parameters():
            p.requires_grad = False
        layers_to_unfreeze = [self.layer3, self.layer4, self.se][-unfreeze_layers:]
        for layer in layers_to_unfreeze:
            for p in layer.parameters():
                p.requires_grad = True

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.se(x)   # (B, 2048, H', W')


# ─────────────────────────────────────────
# 6F. Complete Model
# ─────────────────────────────────────────
class SkinLesionNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, patch_dim=512, bilstm_h=256, trans_d=512):
        super().__init__()
        self.cnn         = GWOCNNBackbone(unfreeze_layers=3)
        self.ngram       = NGramPatchExtractor(2048, patch_dim, n=3)
        self.transformer = TransformerAttentionModel(d=patch_dim, heads=8, layers=4)
        self.anfis       = ANFISLayer(patch_dim, n_rules=16)
        self.dual        = DualStreamFusion(patch_dim, bilstm_h)

        # Feature sizes
        trans_out = patch_dim       # 512
        anfis_out = patch_dim       # 512
        dual_out  = bilstm_h * 2   # 512

        self.fusion = nn.Sequential(
            nn.Linear(trans_out + anfis_out + dual_out, 1024),
            nn.LayerNorm(1024), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),  nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LayerNorm(256),  nn.GELU(), nn.Dropout(0.2),
        )
        self.classifier = nn.Linear(256, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        feat  = self.cnn(x)                     # (B, 2048, H', W')
        seq   = self.ngram(feat)                 # (B, S, 512)
        tr    = self.transformer(seq)            # (B, 512) - CLS token
        an    = self.anfis(tr)                   # (B, 512)
        bl    = self.dual(seq)                   # (B, 512)
        fused = self.fusion(torch.cat([tr,an,bl], dim=-1))
        return self.classifier(fused)            # (B, num_classes)


model = SkinLesionNet(NUM_CLASSES).to(DEVICE)

# Multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"🚀 Using {torch.cuda.device_count()} GPUs with DataParallel!")
    model = nn.DataParallel(model)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model built successfully!")
print(f"   Total params     : {total_params:,}")
print(f"   Trainable params : {trainable_params:,}")


## 🐺 Step 7 — Grey Wolf Optimization (LR + Weight Decay Search)

In [ ]:
def grey_wolf_optimize(model, val_loader, n_wolves=6, max_iter=10):
    """
    GWO searching jointly over:
      pos[0] = log10(learning_rate)  in [-5, -2]
      pos[1] = log10(weight_decay)   in [-6, -2]
    """
    lb = np.array([-5., -6.])
    ub = np.array([-2., -2.])
    pos = np.random.uniform(lb, ub, (n_wolves, 2))

    alpha_s = beta_s = delta_s = float('inf')
    alpha_p = beta_p = delta_p = pos[0].copy()

    counts  = np.bincount([LESION_MAP[d] for d in df["dx"]], minlength=NUM_CLASSES).astype(float)
    cw      = torch.tensor(counts.sum()/(NUM_CLASSES*counts), dtype=torch.float32).to(DEVICE)
    crit    = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)

    def score(p):
        lr, wd = 10**p[0], 10**p[1]
        m_tmp  = type(model.module if hasattr(model,'module') else model)(NUM_CLASSES).to(DEVICE)
        opt    = torch.optim.AdamW(filter(lambda x:x.requires_grad, m_tmp.parameters()), lr=lr, weight_decay=wd)
        m_tmp.train(); total=0; n=0
        with torch.cuda.amp.autocast(enabled=True):
            for imgs, labels in val_loader:
                imgs,labels = imgs.to(DEVICE),labels.to(DEVICE)
                opt.zero_grad()
                loss = crit(m_tmp(imgs), labels)
                loss.backward(); opt.step()
                total += loss.item(); n += 1
                if n >= 5: break
        del m_tmp; torch.cuda.empty_cache()
        return total / max(n, 1)

    print("🐺 Starting Grey Wolf Optimization ...")
    for it in range(max_iter):
        for i in range(n_wolves):
            s = score(pos[i])
            if s < alpha_s: delta_s,delta_p=beta_s,beta_p.copy(); beta_s,beta_p=alpha_s,alpha_p.copy(); alpha_s,alpha_p=s,pos[i].copy()
            elif s < beta_s: delta_s,delta_p=beta_s,beta_p.copy(); beta_s,beta_p=s,pos[i].copy()
            elif s < delta_s: delta_s,delta_p=s,pos[i].copy()

        a = 2 - it*(2/max_iter)
        for i in range(n_wolves):
            new_pos = np.zeros(2)
            for dim in range(2):
                r1,r2=np.random.rand(),np.random.rand(); X1=alpha_p[dim]-(2*a*r1-a)*abs(2*r2*alpha_p[dim]-pos[i,dim])
                r1,r2=np.random.rand(),np.random.rand(); X2=beta_p[dim] -(2*a*r1-a)*abs(2*r2*beta_p[dim] -pos[i,dim])
                r1,r2=np.random.rand(),np.random.rand(); X3=delta_p[dim]-(2*a*r1-a)*abs(2*r2*delta_p[dim]-pos[i,dim])
                new_pos[dim] = np.clip((X1+X2+X3)/3, lb[dim], ub[dim])
            pos[i] = new_pos
        print(f"  iter {it+1:2d}/{max_iter} | loss={alpha_s:.4f} | lr=1e{alpha_p[0]:.3f} | wd=1e{alpha_p[1]:.3f}")

    best_lr = float(10**alpha_p[0])
    best_wd = float(10**alpha_p[1])
    print(f"\n✅ GWO done!  best_lr={best_lr:.6f}  best_wd={best_wd:.6f}")
    return best_lr, best_wd

best_lr, best_wd = grey_wolf_optimize(model, val_loader)


## 🏋️ Step 8 — Training with Mixed Precision & Cosine Scheduler

In [ ]:
# ── Loss & Optimizer ──────────────────────────────────────
counts  = np.bincount([LESION_MAP[d] for d in df["dx"]], minlength=NUM_CLASSES).astype(float)
cw      = torch.tensor(counts.sum()/(NUM_CLASSES*counts), dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.15)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=best_lr, weight_decay=best_wd
)

# Warm-up + cosine annealing
warmup_epochs = 5
def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / (EPOCHS - warmup_epochs)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.cuda.amp.GradScaler()

# ── Run Epoch ──────────────────────────────────────────────
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    all_preds  = []; all_labels = []; all_probs = []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in tqdm(loader, desc="Train" if train else "Eval ", leave=False):
            imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast():
                    out  = model(imgs)
                    loss = criterion(out, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer); scaler.update()
            else:
                with torch.cuda.amp.autocast():
                    out  = model(imgs)
                    loss = criterion(out, labels)

            probs  = F.softmax(out.float(), dim=1)
            preds  = out.argmax(1)
            total_loss += loss.item()
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds  += preds.cpu().tolist()
            all_labels += labels.cpu().tolist()
            all_probs  += probs.cpu().tolist()

    return (total_loss/len(loader), correct/total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))

# ── Training Loop ─────────────────────────────────────────
best_val_acc = 0.0
train_losses, val_losses, train_accs, val_accs = [], [], [], []
history_lrs = []

print(f"🚀 Training for {EPOCHS} epochs on {DEVICE}\n")
start = time.time()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc, *_ = run_epoch(train_loader, train=True)
    vl_loss, vl_acc, vl_pred, vl_true, _ = run_epoch(val_loader, train=False)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    train_losses.append(tr_loss); val_losses.append(vl_loss)
    train_accs.append(tr_acc);   val_accs.append(vl_acc)
    history_lrs.append(current_lr)

    elapsed = time.time() - t0
    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"TrainLoss={tr_loss:.4f} Acc={tr_acc*100:.2f}% | "
          f"ValLoss={vl_loss:.4f} Acc={vl_acc*100:.2f}% | "
          f"LR={current_lr:.2e} | {elapsed:.1f}s")

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        state = model.module.state_dict() if hasattr(model,'module') else model.state_dict()
        torch.save(state, "best_model.pth")
        print(f"  💾 Saved best model — val_acc={vl_acc*100:.2f}%")

total_time = time.time() - start
print(f"\n✅ Training complete in {total_time/60:.1f} min | Best Val Acc: {best_val_acc*100:.2f}%")


## 📊 Step 9 — Test Set Evaluation & All Metrics

In [ ]:
# Load best model
if hasattr(model, 'module'):
    model.module.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
else:
    model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))

_, test_acc, y_pred, y_true, y_prob = run_epoch(test_loader, train=False)

def compute_all_metrics(y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    # Specificity (per class, macro-averaged)
    cm = confusion_matrix(y_true, y_pred)
    spec_list = []
    for i in range(NUM_CLASSES):
        TP = cm[i,i]; FN = cm[i,:].sum()-TP
        FP = cm[:,i].sum()-TP; TN = cm.sum()-TP-FN-FP
        spec_list.append(TN / (TN + FP + 1e-9))
    sensitivity  = rec          # weighted recall = sensitivity
    specificity  = float(np.mean(spec_list))
    return {
        "Accuracy":    acc,
        "Precision":   prec,
        "Recall":      rec,
        "F1-Score":    f1,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
    }

metrics = compute_all_metrics(y_true, y_pred)

print("=" * 62)
print("   FINAL TEST METRICS")
print("=" * 62)
for k, v in metrics.items():
    flag = "✅" if v >= 0.94 else "⚠️ "
    bar  = "█" * int(v * 40)
    print(f"  {flag}  {k:<14s}  {v*100:6.2f}%   {bar}")
print("=" * 62)
print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))


## 📈 Step 10 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Training History — Skin Lesion Classification", fontsize=16, fontweight='bold', y=1.01)

ep = range(1, EPOCHS+1)

# Loss
axes[0].plot(ep, train_losses, 'royalblue', lw=2, label='Train Loss')
axes[0].plot(ep, val_losses,   'tomato',    lw=2, label='Val Loss')
axes[0].fill_between(ep, train_losses, val_losses, alpha=0.08, color='gray')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(ep, [a*100 for a in train_accs], 'royalblue', lw=2, label='Train Acc')
axes[1].plot(ep, [a*100 for a in val_accs],   'tomato',    lw=2, label='Val Acc')
axes[1].axhline(y=94, color='green', linestyle='--', lw=1.5, label='94% target')
axes[1].set(title='Accuracy', xlabel='Epoch', ylabel='Acc (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Learning rate
axes[2].semilogy(ep, history_lrs, 'purple', lw=2)
axes[2].set(title='Learning Rate Schedule', xlabel='Epoch', ylabel='LR (log)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: training_curves.png")


## 🔲 Step 11 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

# Normalised & raw side-by-side
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, data, fmt, title in zip(
    axes,
    [cm, cm.astype('float')/cm.sum(axis=1)[:,None]],
    ['d', '.2f'],
    ['Raw Counts', 'Normalised (Row %)']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, ax=ax, cbar=True, annot_kws={"size":11})
    ax.set_xlabel("Predicted Label", fontsize=13)
    ax.set_ylabel("True Label",      fontsize=13)
    ax.set_title(f"Confusion Matrix — {title}", fontsize=14, fontweight='bold')
    ax.tick_params(axis='both', labelsize=11)

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: confusion_matrix.png")


## 📉 Step 12 — ROC Curves

In [ ]:
y_bin  = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
colors = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Per-class ROC
for i, (cls, col) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    ra = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=col, lw=2.5, label=f"{cls.upper()}  AUC={ra:.3f}")

axes[0].plot([0,1],[0,1],'k--',lw=1.5)
axes[0].set(xlim=[0,1], ylim=[0,1.02],
            xlabel='False Positive Rate', ylabel='True Positive Rate',
            title='ROC — Per Class (One-vs-Rest)')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(alpha=0.3)

# Micro-average ROC
fpr_m, tpr_m, _ = roc_curve(y_bin.ravel(), y_prob.ravel())
auc_m = auc(fpr_m, tpr_m)
axes[1].plot(fpr_m, tpr_m, color='darkorange', lw=3, label=f'Micro-avg AUC={auc_m:.4f}')
axes[1].fill_between(fpr_m, tpr_m, alpha=0.1, color='darkorange')
axes[1].plot([0,1],[0,1],'k--',lw=1.5)
axes[1].set(xlim=[0,1], ylim=[0,1.02],
            xlabel='False Positive Rate', ylabel='True Positive Rate',
            title='ROC — Micro-Average')
axes[1].legend(loc='lower right', fontsize=12)
axes[1].grid(alpha=0.3)

plt.suptitle("ROC Curves — Skin Lesion Classification", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: roc_curves.png")


## 📊 Step 13 — ALL METRICS as Bar Graph (Main Output)

In [ ]:
metric_names  = list(metrics.keys())
metric_values = [v * 100 for v in metrics.values()]

# ── Color palette ──
bar_colors = ['#1E88E5','#43A047','#FB8C00','#E53935','#8E24AA','#00ACC1']

fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('#F4F6F8')

# Draw bars
bars = ax.bar(metric_names, metric_values,
              color=bar_colors, edgecolor='white',
              linewidth=1.8, width=0.55, zorder=3,
              capsize=4, alpha=0.92)

# Gradient overlay (darker top edge)
for bar, col in zip(bars, bar_colors):
    ax.bar(bar.get_x(), bar.get_height(),
           width=bar.get_width(), bottom=0,
           color='none', edgecolor=col, linewidth=2, zorder=4)

# Value labels
for bar, val in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.18,
            f"{val:.2f}%",
            ha='center', va='bottom',
            fontsize=14, fontweight='bold', color='#212121')

# Threshold lines
ax.axhline(y=94, color='#E53935', linestyle='--', linewidth=2.2,
           label='94% threshold', zorder=5)
ax.axhline(y=95, color='#FF6F00', linestyle='--', linewidth=2.2,
           label='95% threshold', zorder=5)

# Shaded target zone
ax.axhspan(94, 101, alpha=0.04, color='green', zorder=1)

# Labels & styling
ax.set_ylim([85, 101])
ax.set_ylabel("Score (%)", fontsize=14, labelpad=10)
ax.set_xlabel("Metric",    fontsize=14, labelpad=10)
ax.set_title(
    "Performance Metrics — Skin Lesion Classification\n"
    "ANFIS + N-Gram + BiLSTM + Transformer Attention + GWO-CNN",
    fontsize=14, fontweight='bold', pad=18
)
ax.tick_params(axis='x', labelsize=13)
ax.tick_params(axis='y', labelsize=12)
ax.grid(axis='y', alpha=0.35, linestyle='--', zorder=0)

# Legend: coloured patches + threshold lines
patches = [mpatches.Patch(color=c, label=n) for c,n in zip(bar_colors, metric_names)]
lines   = [
    plt.Line2D([0],[0], color='#E53935', linestyle='--', lw=2, label='94% threshold'),
    plt.Line2D([0],[0], color='#FF6F00', linestyle='--', lw=2, label='95% threshold'),
]
ax.legend(handles=patches+lines, loc='lower right', fontsize=11,
          framealpha=0.9, edgecolor='gray')

# Spine cleanup
for spine in ['top','right']: ax.spines[spine].set_visible(False)
for spine in ['left','bottom']: ax.spines[spine].set_color('#CCCCCC')

plt.tight_layout()
plt.savefig("metrics_bar_graph.png", dpi=180, bbox_inches='tight')
plt.show()
print("✅ Saved: metrics_bar_graph.png")


## 📊 Step 14 — Per-Class Precision / Recall / F1 Bar Chart

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

prec_c, rec_c, f1_c, sup_c = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES))
)

x = np.arange(NUM_CLASSES)
w = 0.26

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('#F4F6F8')

b1 = ax.bar(x-w,  prec_c*100, w, label='Precision', color='#1E88E5', edgecolor='white', lw=1.5)
b2 = ax.bar(x,    rec_c*100,  w, label='Recall',    color='#43A047', edgecolor='white', lw=1.5)
b3 = ax.bar(x+w,  f1_c*100,   w, label='F1-Score',  color='#FB8C00', edgecolor='white', lw=1.5)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.5,
                f"{h:.1f}", ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.axhline(y=94, color='red',    linestyle='--', lw=1.8, label='94% target')
ax.axhline(y=95, color='orange', linestyle='--', lw=1.8, label='95% target')

ax.set_xticks(x)
ax.set_xticklabels([f"{c.upper()}\n(n={s})" for c,s in zip(CLASS_NAMES,sup_c)], fontsize=11)
ax.set_ylim([0, 110])
ax.set_ylabel("Score (%)", fontsize=13)
ax.set_title("Per-Class Metrics — Precision / Recall / F1", fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='lower right')
ax.grid(axis='y', alpha=0.3, linestyle='--')
for s in ['top','right']: ax.spines[s].set_visible(False)

plt.tight_layout()
plt.savefig("per_class_metrics.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: per_class_metrics.png")


## ✅ Step 15 — Final Summary

In [ ]:
print("=" * 65)
print("  COMPLETE RESULTS SUMMARY — SKIN LESION CLASSIFICATION")
print("=" * 65)
all_pass = True
for k, v in metrics.items():
    flag = "✅ PASS" if v >= 0.94 else "⚠️  BELOW TARGET"
    if v < 0.94: all_pass = False
    bar = '█' * int(v*45) + '░' * (45 - int(v*45))
    print(f"  {k:<14s}  {v*100:6.2f}%  [{bar}]  {flag}")
print("=" * 65)
print(f"  Overall: {'🎉 ALL METRICS ≥ 94%!' if all_pass else '⚠️  Some metrics below target'}")
print("=" * 65)

print("\n📁 Output Files:")
files = ["best_model.pth","training_curves.png","confusion_matrix.png",
         "roc_curves.png","metrics_bar_graph.png","per_class_metrics.png"]
for f in files:
    size = f"{os.path.getsize(f)/1024:.1f} KB" if os.path.exists(f) else "NOT FOUND"
    icon = "✅" if os.path.exists(f) else "❌"
    print(f"  {icon} {f:<35s} {size}")
